# Rolling Horizon

A pathway model can be optimized in three ways:

| | foresight | call |
|---|---|---|
| perfect foresight | all investment periods at once | `esM.optimize()` |
| **rolling horizon** | **a window of consecutive investment periods at a time** | **`fn.rollingHorizonOptimization(esM, numberOfInvestmentPeriodsForRollingHorizon=n)`** |
| myopic foresight | one investment period at a time | `fn.rollingHorizonOptimization(esM, numberOfInvestmentPeriodsForRollingHorizon=1)` |

A rolling horizon optimizes the first window, keeps what it commissioned, moves the window
forward by one investment period and optimizes again. What a window built is handed to the
next one as stock, so each window decides with foresight over its own years only - which is
how an investment decision is actually taken, without knowing the whole future.

This example builds a four period pathway, runs it with a window of two, and shows what the
windows hand each other.

In [1]:
import numpy as np
import pandas as pd

import fine as fn

## The pathway

One region, four investment periods (2020 to 2035 in steps of five years) and two time
steps per year, so that every cell of this notebook solves in a moment.

An electricity demand that grows over the pathway is served by a wind source that gets
cheaper over time, and by a gas plant that is cheap to build but expensive to run.

In [2]:
years = [2020, 2025, 2030, 2035]
hoursPerTimeStep = 4380

# electricity demand in GW per time step and year, growing over the pathway
demandPerYear = {
    2020: [6.0, 4.0],
    2025: [7.0, 5.0],
    2030: [8.5, 6.0],
    2035: [10.0, 7.0],
}


def buildPathway():
    """Return the four period energy system model this example rolls a window along."""
    esM = fn.EnergySystemModel(
        locations={"Region"},
        commodities={"electricity", "naturalGas"},
        commodityUnitsDict={
            "electricity": r"GW$_{el}$",
            "naturalGas": r"GW$_{CH_4,LHV}$",
        },
        numberOfTimeSteps=2,
        hoursPerTimeStep=hoursPerTimeStep,
        costUnit="1e6 Euro",
        numberOfInvestmentPeriods=len(years),
        investmentPeriodInterval=5,
        startYear=years[0],
        lengthUnit="km",
        verboseLogLevel=2,
    )

    esM.add(
        fn.Source(
            esM=esM,
            name="Wind",
            commodity="electricity",
            hasCapacityVariable=True,
            operationRateMax=pd.DataFrame({"Region": [0.6, 0.3]}),
            # wind gets cheaper along the pathway
            investPerCapacity={2020: 1200, 2025: 1000, 2030: 850, 2035: 750},
            opexPerCapacity={year: 20 for year in years},
            interestRate=0.05,
            economicLifetime=20,
            technicalLifetime=20,
        )
    )

    esM.add(
        fn.Source(
            esM=esM,
            name="Gas purchase",
            commodity="naturalGas",
            hasCapacityVariable=False,
            commodityCost=0.03,
        )
    )

    esM.add(
        fn.Conversion(
            esM=esM,
            name="Gas plant",
            physicalUnit=r"GW$_{el}$",
            commodityConversionFactors={"electricity": 1, "naturalGas": -2},
            hasCapacityVariable=True,
            investPerCapacity=600,
            opexPerCapacity=15,
            interestRate=0.05,
            economicLifetime=20,
            technicalLifetime=20,
        )
    )

    esM.add(
        fn.Sink(
            esM=esM,
            name="Demand",
            commodity="electricity",
            hasCapacityVariable=False,
            # demand grows over the pathway. An operation rate is an energy per time
            # step, so a demand of 6 GW over a time step of 4380 hours is 6 * 4380 GWh
            operationRateFix={
                year: pd.DataFrame({"Region": np.array(demand) * hoursPerTimeStep})
                for year, demand in demandPerYear.items()
            },
        )
    )

    return esM

## Rolling the window along the pathway

`numberOfInvestmentPeriodsForRollingHorizon=2` optimizes 2020-2025, then 2025-2030, then
2030-2035. The result is one solved energy system model per window, keyed by the window's
first year.

`timeSeriesAggregation=False` keeps the two time steps of this small example as they are.
For a real model, set it to `True` and describe the clustering through
`timeSeriesAggregationSettings`, which is passed straight to `esM.aggregateTemporally`,
e.g. `{"n_clusters": 7, "period_duration": 24, "cluster": fn.ClusterConfig(method="hierarchical")}`.

In [3]:
results = fn.rollingHorizonOptimization(
    esM=buildPathway(),
    numberOfInvestmentPeriodsForRollingHorizon=2,
    timeSeriesAggregation=False,
)

sorted(results)

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter LogToConsole to value 0


Set parameter OutputFlag to value 1


...

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter LogToConsole to value 0


[2020, 2025, 2030]

## What each window decided

Every window is an ordinary `EnergySystemModel`, so its results are read exactly as after
`esM.optimize()`. Below, the capacity each window commissions in its own first year - which
is the decision that is actually carried out before the window moves on.

In [4]:
def commissioning(esM, year):
    """Return the capacity commissioned per component in one year of one window."""
    summary = esM.getOptimizationSummary("SourceSinkModel", ip=year, outputLevel=0)
    conversion = esM.getOptimizationSummary("ConversionModel", ip=year, outputLevel=0)
    rows = pd.concat([summary, conversion])
    commissioned = rows.xs("commissioning", level="Property")["Region"]
    # components without a capacity variable (the demand, the gas purchase) do not
    # commission anything and are left out
    return commissioned.droplevel("Unit").dropna()


decisions = pd.DataFrame(
    {year: commissioning(results[year], year) for year in sorted(results)}
).round(3)
decisions.columns.name = "window"
decisions

window,2020,2025,2030
Component,,,
Wind,10.0,1.666667,3.333333
Gas plant,1.0,0.5,0.0


## What a window hands to the next one

The capacity a window commissions is not re-decided by the next window: it enters it as
`stockCommissioning`, i.e. as capacity that already exists and only ages from there. That
is what chains the windows into one pathway.

In [5]:
stock2025 = results[2025].getComponent("Wind").stockCommissioning
stock2030 = results[2030].getComponent("Wind").stockCommissioning

print("Wind stock the window starting in 2025 was given:")
print(pd.DataFrame(stock2025).round(3))
print()
print("Wind stock the window starting in 2030 was given:")
print(pd.DataFrame(stock2030).round(3))

Wind stock the window starting in 2025 was given:
          2020
location      
Region    10.0

Wind stock the window starting in 2030 was given:
          2020   2025
location             
Region    10.0  1.667


Stock that has outlived its technical lifetime is dropped on the way, so a window
never inherits capacity that would no longer exist.

## Comparing the windows' costs

Each window discounts its costs onto its own first year, so the `NPVcontribution` of the
window starting in 2030 is expressed in 2030 money and cannot be compared with the one of
the first window. For a rolling horizon run the summary therefore carries a second row,
`NPVcontributionRH`, which is the same value discounted back onto the first year of the
whole pathway - the reference year every window shares.

What a window charges is an annuity for the investment periods it spans. The capacity it
commissions keeps being charged in the following windows, where it arrives as stock, at the
`investPerCapacity` of the year it was originally commissioned in - so no cost falls between
the windows. What a window does *not* see is the cost it commits its successors to beyond
its own last year. That is what limited foresight means, and it is stronger the smaller the
window.

In [6]:
summary2030 = results[2030].getOptimizationSummary(
    "SourceSinkModel", ip=2030, outputLevel=0
)
summary2030.xs("Wind", level="Component").loc[
    ["NPVcontribution", "NPVcontributionRH"]
].round(3)

,,Region
Property,Unit,
NPVcontribution,[1e6 Euro],7382.635841
NPVcontributionRH,[1e6 Euro],4532.297989


## Reading one pathway out of the windows

The windows overlap: 2025 is spanned by the window starting in 2020 *and* by the one
starting in 2025. Adding up every year of every window would therefore count the overlaps
twice. Exactly one window is responsible for each year of the pathway:

* every window for **its own first year** - the decision it actually carries out, and
* the **last window** for **all** of the years it spans, since no later window follows it.

That is the selection `writeExcelOutput=True` writes out, and the one to use when reading
the returned models. Summed over those years, `NPVcontributionRH` is the net present value
of the whole pathway.

In [7]:
def pathwayYears(results):
    """Return one (window, year) pair per year of the pathway, without overlaps."""
    lastWindow = max(results)
    return [
        (window, year)
        for window in sorted(results)
        for year in (
            results[window].investmentPeriodNames if window == lastWindow else [window]
        )
    ]


def npvContributionRH(esM, year, compName):
    """Return one component's NPVcontributionRH in one year of one window."""
    summary = esM.getOptimizationSummary("SourceSinkModel", ip=year, outputLevel=0)
    row = summary.xs((compName, "NPVcontributionRH"), level=("Component", "Property"))
    return row.iloc[0]["Region"]


windPathway = pd.Series(
    {
        year: npvContributionRH(results[window], year, "Wind")
        for window, year in pathwayYears(results)
    },
    name="Wind, NPVcontributionRH [1e6 Euro]",
)
windPathway.index.name = "year"

print(windPathway.round(1).to_string())
print(f"\npathway total: {windPathway.sum():,.1f} 1e6 Euro")

year
2020    5286.5
2025    4737.2
2030    4532.3
2035    4135.6

pathway total: 18,691.7 1e6 Euro


## Myopic foresight

A window of one investment period is myopic foresight: every year is decided on its own,
with no knowledge of the next one at all. It replaces the removed `optimizeSimpleMyopic`.

In this small example the myopic run reaches the same decisions as the two period window
above: the demand of every year is known and grows steadily, so seeing one year further
ahead does not change what is worth building. The two part ways as soon as a decision
only pays off in a later year - a technology that is expensive now and cheap to run, or
a capacity that is needed for a demand peak the myopic window cannot see yet.

Each myopic window holds a single investment period, so FINE points out that the objective
of such a window covers one period while the interval between two periods is five years.
That is what myopic foresight is, and the warning can be ignored here.

In [8]:
myopicResults = fn.rollingHorizonOptimization(
    esM=buildPathway(),
    numberOfInvestmentPeriodsForRollingHorizon=1,
    timeSeriesAggregation=False,
)

myopicDecisions = pd.DataFrame(
    {year: commissioning(myopicResults[year], year) for year in sorted(myopicResults)}
).round(3)
myopicDecisions.columns.name = "window"
myopicDecisions

C:\Users\j.becker\FINE\fine\utils.py:123: UserWarning: Energy system model has only one investment period. However the investment period interval is set to 5. This may results in a higher objective value. 
  warnings.warn(


Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter LogToConsole to value 0


...

Set parameter OutputFlag to value 1


Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter LogToConsole to value 0


window,2020,2025,2030,2035
Component,,,,
Wind,10.0,1.666667,3.333333,3.333333
Gas plant,1.0,0.5,0.0,0.0


## Writing the results out, and continuing an interrupted run

A rolling horizon of a real model runs for a long time, so its windows can be written out
while it goes:

```python
fn.rollingHorizonOptimization(
    esM=buildPathway(),
    numberOfInvestmentPeriodsForRollingHorizon=2,
    writeNetCDFOutput=True,          # one netCDF file, one group per window
    writeExcelOutput=True,           # one Excel file per exported year
    resultExportPath="results",
    scenario_name="myScenario",
)
```

`writeNetCDFOutput=True` writes every window's full model into
`results/myScenario_rollingHorizon.nc`, one group per window, keyed by the window's first
year - the same single file layout perfect foresight uses.

If such a run is interrupted, `resume=True` continues it instead of starting over: before a
window is optimized, its group is looked up in that file and loaded if it is there. A cached
window that was built from other data than the run now being computed is detected and
re-solved rather than trusted, and from the first re-solved window on, every later one is
re-solved too.

```python
fn.rollingHorizonOptimization(
    esM=buildPathway(),
    numberOfInvestmentPeriodsForRollingHorizon=2,
    resume=True,
    resultExportPath="results",
    scenario_name="myScenario",
)
```